In [ ]:
# ses-nota 3. TUR: süslemeli tek saz + düet sentetik ses ve gerçek kayıt -- TEK HÜCRE. Çalışma zamanı: A100.
# 2. aşamanın (ince ayar) ağırlıklarından devam eder; etiketler hep yazılı nota.
AD = 'tur3-v1'        # koşu adı: runs/ic/<tarih>-<AD>/
BASLANGIC = None      # None: 2. aşamanın (ince-ayar) son koşusunun son.pt'si · ya da bir .pt yolu
ADIM = 15000
DEVAM = None          # oturum kesilirse: 'son' (bu 3. tur koşusunun son checkpoint'i)
KAPAT = True         # bitince Drive'ı kaydedip çalışma zamanını kapat (kota harcamasın)
# ----------------------------------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil, zipfile, json, glob
from pathlib import Path
DRIVE = '/content/drive/MyDrive/ses_nota'
KOD = '/content/ses-nota'
shutil.rmtree(KOD, ignore_errors=True)
zipfile.ZipFile(f'{DRIVE}/kod/kod.zip').extractall(KOD)
os.environ['SES_NOTA_HOMR'] = f'{KOD}/homr_kopya'
!pip -q install librosa soundfile musicxml==1.4 mir_eval
sys.path[:0] = [f'{KOD}/model', f'{KOD}/model/colab']
import defter
_kapat = KAPAT
try:
    KOK = f'{DRIVE}/runs/ic'
    runs = sorted((p for p in Path(KOK).glob('*') if (p / 'ayar.json').exists()), key=lambda p: p.stat().st_mtime)
    if BASLANGIC is None:
        birinci = [p for p in runs if 'ince-ayar' in p.name and (p / 'son.pt').exists()]
        BASLANGIC = str(birinci[-1] / 'son.pt')
    print('başlangıç ağırlıkları:', BASLANGIC)
    devam = DEVAM
    if DEVAM == 'son':
        ince = [p for p in runs if 'tur3' in p.name]
        pts = sorted(glob.glob(f'{ince[-1]}/kontrol_*.pt')) if ince else []
        devam = pts[-1] if pts else None
        print('devam:', devam)
    print(defter.copy_shards(f'{DRIVE}/veri', '/content/veri',
                             ['acik/train', 'ic/ddsp/train', 'ic/ornek/train', 'acik3/train', 'ic/duet/train',
                              'acik/val', 'acik3/val', 'ic/gercek/train', 'ic/gercek/val']))
    cfg = json.loads(Path(f'{KOD}/model/ayarlar/tur3.json').read_text(encoding='utf-8'))
    cfg['ad'] = AD
    cfg['egitim']['baslangic'] = BASLANGIC
    cfg['egitim']['adim'] = ADIM
    cfg['veri']['isci'] = min(8, os.cpu_count() or 2)
    kod = defter.train(KOD, cfg, f'/content/ayar_{AD}.json', resume=devam)
    run = defter.latest_run(KOK)
    print('koşu:', run, '| çıkış kodu:', kod)
    if kod == 0:
        KONTROL = str(run / 'son.pt')
        symbtr = f'{DRIVE}/veri/acik/symbtr_txt'
        sonuc = defter.evaluate(KOD, KONTROL, ['sentetik', 'tanbur', 'kayit'], str(Path(KONTROL).parent / 'sinav'),
                                synthetic_shards=[f'{DRIVE}/veri/acik/test'], karar='verilen',
                                symbtr_txt=symbtr if os.path.isdir(symbtr) else None,
                                tanbur_dir=f'{DRIVE}/veri/ic/sinav-tanbur', ic_out_root=f'{DRIVE}/runs/ic/sinav',
                                kayit_dir=f'{DRIVE}/veri/ic/sinav-kayit',
                                ince_ayar_bolum=f'{DRIVE}/veri/ic/ince-ayar/bolum.json')
        print(json.dumps(sonuc, ensure_ascii=False, indent=1)[:6000])
except KeyboardInterrupt:  # stopped by hand: keep the runtime
    _kapat = False
    raise
finally:
    print('BİTTİ.')
    if _kapat:
        print('Drive kaydediliyor, çalışma zamanı kapatılıyor (KAPAT = True)...')
        drive.flush_and_unmount()
        from google.colab import runtime
        runtime.unassign()
    else:
        print('Runtime → Disconnect and delete runtime (A100 boşta da kota harcar).')